# PhantomWiki Dataset Exploration

PhantomWiki generates synthetic Wikipedia-style biographies with associated reasoning questions. It uses Prolog internally to derive answers.

**Prerequisites:**
1. Install SWI-Prolog from https://www.swi-prolog.org/download/stable (Windows installer)
   - During install, let it add itself to the PATH
   - Restart your terminal / kernel after installation
2. `pip install phantom-wiki` (already done if you're using the project venv)

## 1. Generate a small sample dataset

In [ ]:
import phantom_wiki as pw

OUTPUT_DIR = "./data/phantom_wiki_sample"

pw.generate_dataset(
    output_dir=OUTPUT_DIR,
    seed=42,
    num_family_trees=1,
    max_family_tree_size=10,   # small universe: ~10 people
    max_family_tree_depth=3,
    question_depth=2,          # shallow questions only
    num_questions_per_type=3,  # 3 questions per template
    article_format="json",
    question_format="json",
    quiet=False,
)

## 2. Inspect generated files

In [ ]:
import os

for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

## 3. Articles

In [ ]:
import json

with open(os.path.join(OUTPUT_DIR, "articles.json")) as f:
    articles = json.load(f)

print(f"{len(articles)} articles generated\n")
print("Fields per article:", list(articles[0].keys()))

In [ ]:
# Show a sample article
sample = articles[0]
print(f"=== {sample['title']} ===")
print()
print(sample["article"])
print()
print("Raw facts:")
for fact in sample["facts"]:
    print(" ", fact)

## 4. Questions

In [ ]:
with open(os.path.join(OUTPUT_DIR, "questions.json")) as f:
    questions = json.load(f)

print(f"{len(questions)} questions generated\n")
print("Fields per question:", list(questions[0].keys()))

In [ ]:
import pandas as pd

df = pd.DataFrame(questions)
df[["question", "answer", "difficulty", "is_aggregation_question"]].head(10)

In [ ]:
# Distribution of question types and difficulties
print("Questions by type:")
print(df["type"].value_counts().sort_index().to_string())
print()
print("Difficulty stats:")
print(df["difficulty"].describe())

In [ ]:
# Show a few example QA pairs
for q in questions[:5]:
    print(f"Q: {q['question']}")
    print(f"A: {q['answer']}")
    print(f"   difficulty={q['difficulty']}, aggregation={q['is_aggregation_question']}")
    print()

## 5. Prolog query behind the scenes

In [ ]:
# Each question has the underlying Prolog query that produced the answer
q = questions[0]
print(f"Question : {q['question']}")
print(f"Answer   : {q['answer']}")
print(f"Prolog   : {q['prolog']['query']}")